# Import libraries

In [1]:
# Changes to all modules will automatically be applied when any cell runs. 
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
import numpy as np

import optuna

from pathlib import Path
import sys

sys.path.append(
    str(Path('..', 'utils_functionality', 'split_utils'))
)
from split_tools import get_train_test

sys.path.append(
    str(Path('..', 'utils_functionality', 'models'))
)

from modelling4_utils import (
    BetaVAEncoder,
    MLPipeline,
)

from catboost import CatBoostClassifier
from statsmodels.api import Logit

from sklearn.decomposition import PCA

from sklearn.metrics import (
    make_scorer,
    f1_score,
    mean_squared_error
)

import optuna
from functools import partial



In [ ]:

# To simplify data extraction, initialize ml_pipe
target = 'splashing'
model_postfix = 'vae'
ml_pipe = MLPipeline(
    target=target,
    estimator=CatBoostClassifier,
    estimator_params={'verbose': False,},
    # post_transformer=PCA,
    # post_transformer_params={'n_components': 6},
    post_transformer=BetaVAEncoder,
    post_transformer_params={
        'VAE_params': {
            'latent_dim': 5,
            'hidden_dim': 256,
        },
        'verbose': True,
        'learning_rate': 5e-2,
        'early_stopping': True,
        'early_stopping_patience': 30,
        # scheduler_class=None,
        'beta_warmup_steps': 200,
        'beta_start': 0.0,
        'beta_end': 1.,
        'max_epochs': 1000,
    },
    features_to_drop=(
        'sign_sedimentation_Re',
        'sign_sedimentation_Stk',
        'sign_particle_droplet_diameter_ratio',
    ),
    cv_folds=7,
    model_postfix=model_postfix,
    verbose=True,
)

# X_train = ml_pipe.train.drop(columns=[target])
# X_test = ml_pipe.test.drop(columns=[target])
# display(X_train.info())
# display(X_test.info())

ml_pipe.run(
    save_model_and_metrics=False,
    cv_verbose=False,
)

Оптимизируем параметры VAE encoder, прибегнув к кросс-валидационной оценке MSE при реконструкции исходных признаков

In [ ]:
vae_encoder = BetaVAEncoder(
    VAE_params={
        'latent_dim': 4,
        'hidden_dim': 128,
    },
    verbose=True,
    learning_rate=5e-2,
    early_stopping=True,
    early_stopping_patience=50,
    # scheduler_class=None,
    beta_end=10,
    max_epochs=200,
)
vae_encoder.fit(X_train)

Z_test = vae_encoder.transform(X_test)
X_test_reconstr = vae_encoder.inverse_transform(Z_test)

MSE_test = mean_squared_error(X_test, X_test_reconstr)
print(f'MSE_test: {MSE_test}')
